<a href="https://colab.research.google.com/github/kyeeun7706-coder/e-waste-location-optimization/blob/main/%EC%A0%84%EC%A2%85%EC%84%A4_0330.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

라이브러리 설치

In [3]:
!pip install geopandas osmnx networkx rasterio rasterstats shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 2.1 MB/s eta 0:00:00


라이브러리 임포트 + 드라이브 마운트

In [5]:
import pandas as pd
import geopandas as gpd
import numpy as np
import osmnx as ox
import networkx as nx
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import Point

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


파일 경로 설정

In [6]:
BASE = '/content/drive/MyDrive/전종설'

# 지표 1 - 행정동 경계 (폴더 안 .shp)
DONG_SHP = f'{BASE}/지표1_행정구역경계/bnd_dong_11180_2025_2Q.shp'

# 지표 1 - 연령별 인구 CSV
POP_CSV = f'{BASE}/지표1_202412_202512_금천구_연령별인구현황_연간.csv'

# 지표 2 - DEM 래스터
DEM_IMG = f'{BASE}/지표2_서울dem.img'

# 지표 2 - 격자별 고령인구 (폴더 안 .shp)
GRID_OLD_SHP = f'{BASE}/지표2_금천구격자별고령인구수/vl_blk.shp'

# 지표 2 - 격자별 총인구 (폴더 안 .shp)
GRID_TOT_SHP = f'{BASE}/지표2_금천구격자별총인구수/vl_blk.shp'

# 지표 3 - 수거함 8개 위치
BINS_CSV = f'{BASE}/지표3_금천구폐가전수거함8개위치.csv'



---



파일 정상 로드 확인

In [7]:
# 행정동 경계
dong = gpd.read_file(DONG_SHP)
print("행정동 경계 컬럼:", dong.columns.tolist())
print(dong.head(3))

행정동 경계 컬럼: ['BASE_DATE', 'ADM_CD', 'ADM_NM', 'geometry']
  BASE_DATE    ADM_CD ADM_NM  \
0  20250630  11180510    가산동   
1  20250630  11180520   독산1동   
2  20250630  11180530   독산2동   

                                            geometry  
0  POLYGON ((945134.075 1943182.19, 945138.213 19...  
1  POLYGON ((946851.545 1942312.478, 946849.923 1...  
2  POLYGON ((947770.689 1941031.612, 947770.422 1...  


In [8]:
# 연령별 인구 CSV
pop = pd.read_csv(POP_CSV, encoding='EUC-KR')
print("인구 CSV 컬럼:", pop.columns.tolist())
print(pop.head(3))

인구 CSV 컬럼: ['행정구역', '2024년_거주자_총인구수', '2024년_거주자_연령구간인구수', '2024년_거주자_65~69세', '2024년_거주자_70~74세', '2024년_거주자_75~79세', '2024년_거주자_80~84세', '2024년_거주자_85~89세', '2024년_거주자_90~94세', '2024년_거주자_95~99세', '2024년_거주자_100세 이상', '2024년_남_거주자_총인구수', '2024년_남_거주자_연령구간인구수', '2024년_남_거주자_65~69세', '2024년_남_거주자_70~74세', '2024년_남_거주자_75~79세', '2024년_남_거주자_80~84세', '2024년_남_거주자_85~89세', '2024년_남_거주자_90~94세', '2024년_남_거주자_95~99세', '2024년_남_거주자_100세 이상', '2024년_여_거주자_총인구수', '2024년_여_거주자_연령구간인구수', '2024년_여_거주자_65~69세', '2024년_여_거주자_70~74세', '2024년_여_거주자_75~79세', '2024년_여_거주자_80~84세', '2024년_여_거주자_85~89세', '2024년_여_거주자_90~94세', '2024년_여_거주자_95~99세', '2024년_여_거주자_100세 이상', '2025년_거주자_총인구수', '2025년_거주자_연령구간인구수', '2025년_거주자_65~69세', '2025년_거주자_70~74세', '2025년_거주자_75~79세', '2025년_거주자_80~84세', '2025년_거주자_85~89세', '2025년_거주자_90~94세', '2025년_거주자_95~99세', '2025년_거주자_100세 이상', '2025년_남_거주자_총인구수', '2025년_남_거주자_연령구간인구수', '2025년_남_거주자_65~69세', '2025년_남_거주자_70~74세', '2025년_남_거주자_75~79세', '2025년_남_거주자_80~84세', '2025년_남_

In [9]:
# 격자별 고령인구
grid_old = gpd.read_file(GRID_OLD_SHP)
print("격자 고령인구 컬럼:", grid_old.columns.tolist())
print(grid_old.head(3))

격자 고령인구 컬럼: ['gid', 'lbl', 'val', 'geometry']
            gid   lbl  val                                           geometry
0  ë¤ì¬444429  None  NaN  POLYGON ((944400 1942900, 944400 1943000, 9445...
1  ë¤ì¬444430  None  NaN  POLYGON ((944400 1943000, 944400 1943100, 9445...
2  ë¤ì¬444431  None  NaN  POLYGON ((944400 1943100, 944400 1943200, 9445...


In [10]:
# 격자별 총인구
grid_tot = gpd.read_file(GRID_TOT_SHP)
print("격자 총인구 컬럼:", grid_tot.columns.tolist())
print(grid_tot.head(3))

격자 총인구 컬럼: ['gid', 'lbl', 'val', 'geometry']
            gid   lbl  val                                           geometry
0  ë¤ì¬444429  None  NaN  POLYGON ((944400 1942900, 944400 1943000, 9445...
1  ë¤ì¬444430  None  NaN  POLYGON ((944400 1943000, 944400 1943100, 9445...
2  ë¤ì¬444431  None  NaN  POLYGON ((944400 1943100, 944400 1943200, 9445...


In [11]:
# DEM
with rasterio.open(DEM_IMG) as src:
    print("DEM CRS:", src.crs)
    print("DEM 해상도:", src.res)
    print("DEM 크기:", src.width, "x", src.height)

DEM CRS: EPSG:5179
DEM 해상도: (90.0, 90.0)
DEM 크기: 253 x 316


In [12]:
# 수거함 위치
bins_df = pd.read_csv(BINS_CSV)
print("수거함 컬럼:", bins_df.columns.tolist())
print(bins_df)

수거함 컬럼: ['id', 'name', 'lat', 'lon']
   id   name        lat         lon
0   1  수거함_1  37.469710  126.898966
1   2  수거함_2  37.465816  126.897190
2   3  수거함_3  37.462436  126.898027
3   4  수거함_4  37.463940  126.891584
4   5  수거함_5  37.462466  126.890374
5   6  수거함_6  37.457927  126.897581
6   7  수거함_7  37.460229  126.886538
7   8  수거함_8  37.451069  126.897925




---



### 🚨 지표2_총인구/고령인구 한글 인코딩 깨짐 이슈 발생

격자 인코딩 문제 해결 + 데이터 확인

In [13]:
# dbf 파일을 직접 읽어서 인코딩 문제 해결
grid_old = gpd.read_file(GRID_OLD_SHP, encoding='cp949')
grid_tot = gpd.read_file(GRID_TOT_SHP, encoding='cp949')

print("격자 고령인구 샘플:")
print(grid_old.head(5))
print("\nval 컬럼 null 개수:", grid_old['val'].isna().sum())
print("val 컬럼 유효값 개수:", grid_old['val'].notna().sum())
print("\n격자 총인구 샘플:")
print(grid_tot.head(5))

격자 고령인구 샘플:
        gid   lbl  val                                           geometry
0  떎궗444429  None  NaN  POLYGON ((944400 1942900, 944400 1943000, 9445...
1  떎궗444430  None  NaN  POLYGON ((944400 1943000, 944400 1943100, 9445...
2  떎궗444431  None  NaN  POLYGON ((944400 1943100, 944400 1943200, 9445...
3  떎궗445427  None  NaN  POLYGON ((944500 1942700, 944500 1942800, 9446...
4  떎궗445428  None  NaN  POLYGON ((944500 1942800, 944500 1942900, 9446...

val 컬럼 null 개수: 715
val 컬럼 유효값 개수: 709

격자 총인구 샘플:
        gid   lbl  val                                           geometry
0  떎궗444429  None  NaN  POLYGON ((944400 1942900, 944400 1943000, 9445...
1  떎궗444430  None  NaN  POLYGON ((944400 1943000, 944400 1943100, 9445...
2  떎궗444431  None  NaN  POLYGON ((944400 1943100, 944400 1943200, 9445...
3  떎궗445427  None  NaN  POLYGON ((944500 1942700, 944500 1942800, 9446...
4  떎궗445428  None  NaN  POLYGON ((944500 1942800, 944500 1942900, 9446...


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp949 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp949 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(


val 값 실제 확인

In [14]:
# NaN 아닌 행만 보기
print("고령인구 격자 유효값 샘플:")
print(grid_old[grid_old['val'].notna()].head(10))

print("\n총인구 격자 유효값 샘플:")
print(grid_tot[grid_tot['val'].notna()].head(10))

print("\n고령인구 val 통계:")
print(grid_old['val'].describe())

print("\n총인구 val 통계:")
print(grid_tot['val'].describe())

고령인구 격자 유효값 샘플:
         gid    lbl   val                                           geometry
14  떎궗446430    N/A   0.0  POLYGON ((944600 1943000, 944600 1943100, 9447...
21  떎궗447427    N/A   0.0  POLYGON ((944700 1942700, 944700 1942800, 9448...
23  떎궗447429    N/A   0.0  POLYGON ((944700 1942900, 944700 1943000, 9448...
33  떎궗448427   6.00   6.0  POLYGON ((944800 1942700, 944800 1942800, 9449...
42  떎궗449422  70.00  70.0  POLYGON ((944900 1942200, 944900 1942300, 9450...
43  떎궗449423  11.00  11.0  POLYGON ((944900 1942300, 944900 1942400, 9450...
56  떎궗450420    N/A   0.0  POLYGON ((945000 1942000, 945000 1942100, 9451...
58  떎궗450422    N/A   0.0  POLYGON ((945000 1942200, 945000 1942300, 9451...
61  떎궗450425    N/A   0.0  POLYGON ((945000 1942500, 945000 1942600, 9451...
65  떎궗450429    N/A   0.0  POLYGON ((945000 1942900, 945000 1943000, 9451...

총인구 격자 유효값 샘플:
         gid     lbl    val                                           geometry
14  떎궗446430     N/A    0.0  POLYGON ((944

좌표계 확인(모두 같아야 함)

In [15]:
print("행정동 경계 CRS:", dong.crs)
print("격자 고령인구 CRS:", grid_old.crs)
print("격자 총인구 CRS:", grid_tot.crs)
print("DEM CRS: EPSG:5179")

행정동 경계 CRS: EPSG:5179
격자 고령인구 CRS: EPSG:5179
격자 총인구 CRS: EPSG:5179
DEM CRS: EPSG:5179




---



### 지표 1 준비
인구 CSV 전처리

In [16]:
# 2024년 데이터만 사용, 필요한 컬럼만 추출
pop_cols = [
    '행정구역',
    '2024년_거주자_총인구수',
    '2024년_거주자_65~69세',
    '2024년_거주자_70~74세',
    '2024년_거주자_75~79세',
    '2024년_거주자_80~84세',
    '2024년_거주자_85~89세',
    '2024년_거주자_90~94세',
    '2024년_거주자_95~99세',
    '2024년_거주자_100세 이상',
]
pop_sub = pop[pop_cols].copy()

# 금천구 전체 행(첫 번째 행) 제외, 동 단위만 남기기
pop_sub = pop_sub[pop_sub['행정구역'].str.contains('동')]

# 숫자 컬럼 쉼표 제거 후 정수 변환
for col in pop_cols[1:]:
    pop_sub[col] = pop_sub[col].astype(str).str.replace(',', '').str.strip()
    pop_sub[col] = pd.to_numeric(pop_sub[col], errors='coerce')

# 65세 이상 합산
age_cols = [c for c in pop_cols if '세' in c or '이상' in c]
pop_sub['고령인구'] = pop_sub[age_cols].sum(axis=1)
pop_sub['총인구'] = pop_sub['2024년_거주자_총인구수']

# 행정동명 정리 (괄호 앞 이름만 추출)
pop_sub['행정동명'] = pop_sub['행정구역'].str.extract(r'금천구\s+(.+?)\s*\(')

# 결과 확인
print(pop_sub[['행정동명', '총인구', '고령인구']].to_string())

     행정동명    총인구  고령인구
1     가산동  24665  3038
2   독산제1동  46980  6892
3   독산제2동  17346  4057
4   독산제3동  22793  5314
5   독산제4동  14047  3311
6   시흥제1동  31476  7051
7   시흥제2동  20164  4763
8   시흥제3동  10101  2678
9   시흥제4동  18673  4846
10  시흥제5동  17564  4846


행정동 경계와 인구 데이터 결합 확인

In [17]:
# 행정동명 일치 여부 확인
print("경계 파일 동명:", dong['ADM_NM'].tolist())
print("\nCSV 동명:", pop_sub['행정동명'].tolist())

경계 파일 동명: ['가산동', '독산1동', '독산2동', '독산3동', '독산4동', '시흥1동', '시흥2동', '시흥3동', '시흥4동', '시흥5동']

CSV 동명: ['가산동', '독산제1동', '독산제2동', '독산제3동', '독산제4동', '시흥제1동', '시흥제2동', '시흥제3동', '시흥제4동', '시흥제5동']




---

